# 06 — XGBoost E 가중치 + LR E 50:50 + 팀 공통 홀드아웃

공통 CSV를 **재생성하지 않고** 읽습니다. 경로는 노트북 실행 위치와 무관하게 자동 탐색합니다.

**중요:** 홀드아웃 오염을 줄이기 위해 모델 비교용 CV는 개발셋 4,960명에서만 수행합니다. 기존 전체 6,201명 CV 결과와 수치가 직접 같지는 않습니다. 최종 전체 재학습은 통과 후에만 수행합니다.


In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from xgboost import XGBClassifier

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p/'src'/'config.py').is_file() and (p/'scripts'/'fixed_holdout_manifest.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('src/config.py가 있는 프로젝트 폴더에서 실행하세요.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 노트북의 현재 작업 폴더와 무관하게 프로젝트 스크립트 로딩
import importlib.util
manifest_path = ROOT / 'scripts' / 'fixed_holdout_manifest.py'
spec = importlib.util.spec_from_file_location('fixed_holdout_manifest', manifest_path)
holdout_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(holdout_module)
assert holdout_module.ROOT.resolve() == ROOT.resolve(), '스크립트의 ROOT 경로를 확인하세요.'
from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, save_predictions, log_result
from src.features import mutation_count, mutation_type_counts, amino_acid_pair_counts, gene_truncation_flags

set_seed()
RUN_FINAL_TRAIN = False   # 홀드아웃 검증 결과를 확인한 뒤 True로 바꾸세요.
SAVE_ARTIFACTS = False    # 최종 재학습 완료 후 True로 바꾸세요.
print('ROOT', ROOT, '| XGBoost', __import__('xgboost').__version__)
print('홀드아웃 스크립트:', manifest_path)

ROOT C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | XGBoost 3.4.1
홀드아웃 스크립트: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\scripts\fixed_holdout_manifest.py


In [2]:
def select_xgb_device():
    try:
        probe = XGBClassifier(n_estimators=1, max_depth=1, tree_method='hist', device='cuda',
                              objective='binary:logistic', eval_metric='logloss', verbosity=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            probe.fit(np.array([[0.],[1.],[2.],[3.]],dtype=np.float32), np.array([0,0,1,1]))
        actual = json.loads(probe.get_booster().save_config())['learner']['generic_param']['device']
        if str(actual).startswith('cuda'):
            return 'cuda'
    except Exception as exc:
        print('CUDA 확인 실패:', repr(exc))
    return 'cpu'

DEVICE = select_xgb_device()
print('XGBoost device:', DEVICE)

XGBoost device: cuda


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert len(le.classes_) == 26
# 공통 홀드아웃 먼저 읽기: CV는 개발 데이터에서만 실행합니다.
dev_idx, hold_idx = holdout_module.verify()
assert len(dev_idx) + len(hold_idx) == len(y)
assert not np.intersect1d(dev_idx, hold_idx).size
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]
folds = get_folds(y_dev, X=X_dev_raw, classes=le.classes_)  # 팀 함수 변경 금지
print('train', X_raw.shape, 'test', X_test_raw.shape, 'classes', len(le.classes_))

def make_E(X):
    # 시윤님 02_features E와 동일한 피처 순서/변환
    parts = [binarize_mutations(X), np.log1p(mutation_count(X)),
             np.log1p(mutation_type_counts(X)), np.log1p(amino_acid_pair_counts(X)),
             gene_truncation_flags(X)]
    out = pd.concat(parts, axis=1).astype(np.float32)
    assert not out.columns.has_duplicates and out.index.equals(X.index)
    assert np.isfinite(out.to_numpy()).all()
    return out

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns)
assert X_E.shape[1] == 9174, f'E 피처 개수 불일치: {X_E.shape[1]}'
print('E', X_E.shape, X_test_E.shape)
X_dev_E = X_E.iloc[dev_idx].reset_index(drop=True)


VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
train (6201, 4384) test (2546, 4384) classes 26
E (6201, 9174) (2546, 9174)


## 1. 팀 Group CV — 가중치 이외의 XGBoost 설정 고정

가중치 `1/sqrt(학습 Fold의 클래스별 빈도)`를 **해당 Fold의 학습 라벨로만** 계산합니다. 평균 1 정규화는 가중치 전체 크기를 맞추기 위한 선택이며, 과거 LB 0.3130 실험의 정확한 재현이라고 주장하지 않습니다.

In [4]:
K = len(le.classes_)
common = dict(n_estimators=250, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, objective='multi:softprob', num_class=K,
              eval_metric='mlogloss', tree_method='hist', device=DEVICE,
              random_state=SEED, n_jobs=-1)
xgb = XGBClassifier(max_depth=3, **common)
lr = LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000, solver='lbfgs')

def weight_array(y_part):
    y_part = np.asarray(y_part, dtype=int)
    counts = np.bincount(y_part, minlength=K)
    assert np.all(counts > 0), '학습 구간에 없는 클래스가 있습니다'
    weights = 1.0 / np.sqrt(counts[y_part])
    return (weights / weights.mean()).astype(np.float32)

def weighted_fit_params(y_part):
    return {'sample_weight': weight_array(y_part)}

print('무가중 XGBoost Group CV')
cv_xgb_base = run_cv(xgb, X_dev_E, y_dev, folds=folds)
print('가중 XGBoost Group CV')
cv_xgb_weighted = run_cv(xgb, X_dev_E, y_dev, folds=folds, fit_params_fn=weighted_fit_params)
print('LR E Group CV (시윤님 설정, 동일 Fold)')
# LR의 FunctionTransformer는 make_E를 csr_matrix로 바꾸는 역할: 미리 만든 E의 희소행렬과 동등
X_lr = pd.DataFrame.sparse.from_spmatrix(csr_matrix(X_dev_E.to_numpy()), index=X_dev_E.index, columns=X_dev_E.columns)
cv_lr = run_cv(lr, X_lr, y_dev, folds=folds)


무가중 XGBoost Group CV


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [11:13:01] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.4767
fold 2: Macro F1 = 0.4472
fold 3: Macro F1 = 0.4724
fold 4: Macro F1 = 0.4737
fold 5: Macro F1 = 0.4786
CV Macro F1 = 0.4697 (std 0.0115)
가중 XGBoost Group CV
fold 1: Macro F1 = 0.5049
fold 2: Macro F1 = 0.4753
fold 3: Macro F1 = 0.4806
fold 4: Macro F1 = 0.4810
fold 5: Macro F1 = 0.5048
CV Macro F1 = 0.4893 (std 0.0128)
LR E Group CV (시윤님 설정, 동일 Fold)
fold 1: Macro F1 = 0.5025
fold 2: Macro F1 = 0.4644
fold 3: Macro F1 = 0.4740
fold 4: Macro F1 = 0.4757
fold 5: Macro F1 = 0.4889
CV Macro F1 = 0.4811 (std 0.0132)


In [5]:
def checked_proba(p, n):
    p = np.asarray(p, dtype=np.float64)
    assert p.shape == (n, K) and np.isfinite(p).all()
    assert np.allclose(p.sum(axis=1), 1, atol=1e-4)
    return p

base_oof = checked_proba(cv_xgb_base['oof_proba'], len(y_dev))
weighted_oof = checked_proba(cv_xgb_weighted['oof_proba'], len(y_dev))
lr_oof = checked_proba(cv_lr['oof_proba'], len(y_dev))
ensemble_base_oof = (base_oof + lr_oof) / 2
ensemble_weighted_oof = (weighted_oof + lr_oof) / 2

def fold_metrics(proba):
    scores = [macro_f1(y_dev[va], np.argmax(proba[va],axis=1)) for _,va in folds]
    return {'mean':float(np.mean(scores)), 'std':float(np.std(scores)),
            'oof':macro_f1(y_dev,np.argmax(proba,axis=1)), 'scores':scores}

cv_ens_base = fold_metrics(ensemble_base_oof)
cv_ens_weighted = fold_metrics(ensemble_weighted_oof)
summary = pd.DataFrame([
    {'model':'XGB base','cv_mean':cv_xgb_base['mean'],'cv_std':cv_xgb_base['std'],'oof':macro_f1(y_dev,cv_xgb_base['oof'])},
    {'model':'XGB weighted','cv_mean':cv_xgb_weighted['mean'],'cv_std':cv_xgb_weighted['std'],'oof':macro_f1(y_dev,cv_xgb_weighted['oof'])},
    {'model':'LR E','cv_mean':cv_lr['mean'],'cv_std':cv_lr['std'],'oof':macro_f1(y_dev,cv_lr['oof'])},
    {'model':'XGB base + LR 50:50','cv_mean':cv_ens_base['mean'],'cv_std':cv_ens_base['std'],'oof':cv_ens_base['oof']},
    {'model':'XGB weighted + LR 50:50','cv_mean':cv_ens_weighted['mean'],'cv_std':cv_ens_weighted['std'],'oof':cv_ens_weighted['oof']},
])
display(summary)
cv_delta = cv_ens_weighted['mean'] - cv_ens_base['mean']
print('앙상블 CV 개선:',round(cv_delta,5))

,model,cv_mean,cv_std,oof
0,XGB base,0.469733,0.011456,0.474431
1,XGB weighted,0.489291,0.012839,0.493004
2,LR E,0.481087,0.013241,0.484412
3,XGB base + LR 50:50,0.492355,0.011569,0.495962
4,XGB weighted + LR 50:50,0.501080,0.014111,0.504999


앙상블 CV 개선: 0.00872


## 2. 고정 홀드아웃 검증 및 동일 개발셋 모델 비교

분할 파일은 3번 데이터 준비 셀에서 이미 검증했습니다. 이 셀은 같은 개발셋에서 A/B 모델을 학습해 홀드아웃을 **한 번** 평가합니다.


In [6]:

# 저장된 공통 환자 ID 분할을 불러오며 새 분할은 생성하지 않습니다.
# 분할은 앞의 데이터 준비 셀에서 검증/고정했습니다.
assert len(dev_idx) == 4960 and len(hold_idx) == 1241
raw_groups = pd.util.hash_pandas_object(X_raw.fillna('<MISSING>').astype(str), index=False).to_numpy()
assert not set(raw_groups[dev_idx]) & set(raw_groups[hold_idx])
assert len(np.intersect1d(dev_idx, hold_idx)) == 0
assert len(dev_idx) + len(hold_idx) == len(y)
print('개발:',len(dev_idx),'홀드아웃:',len(hold_idx),'비율:',round(len(hold_idx)/len(y),4))

X_dev, X_hold = X_E.iloc[dev_idx], X_E.iloc[hold_idx]
y_dev, y_hold = y[dev_idx], y[hold_idx]

base_hold_model = clone(xgb).fit(X_dev, y_dev)
weighted_hold_model = clone(xgb).fit(X_dev, y_dev, sample_weight=weight_array(y_dev))
# LR E는 02_features의 희소 입력과 동일
lr_hold_model = clone(lr).fit(csr_matrix(X_dev.to_numpy()), y_dev)

for model in (base_hold_model, weighted_hold_model, lr_hold_model):
    assert np.array_equal(model.classes_, np.arange(K))

p_base = checked_proba(base_hold_model.predict_proba(X_hold), len(hold_idx))
p_weighted = checked_proba(weighted_hold_model.predict_proba(X_hold), len(hold_idx))
p_lr = checked_proba(lr_hold_model.predict_proba(csr_matrix(X_hold.to_numpy())), len(hold_idx))

hold_base = macro_f1(y_hold, np.argmax((p_base+p_lr)/2, axis=1))
hold_weighted = macro_f1(y_hold, np.argmax((p_weighted+p_lr)/2, axis=1))
hold_delta = hold_weighted - hold_base
cv_pass = cv_delta > 0
hold_pass = hold_delta >= -0.02 - 1e-12
candidate_pass = cv_pass and hold_pass

display(pd.DataFrame([
    {'candidate':'기존 XGB + LR','cv_mean':cv_ens_base['mean'],'holdout_macro_f1':hold_base},
    {'candidate':'가중 XGB + LR','cv_mean':cv_ens_weighted['mean'],'holdout_macro_f1':hold_weighted},
]))
print(f'CV 변화 {cv_delta:+.4f} | 홀드아웃 변화 {hold_delta:+.4f}')
print('제출 후보 판단:', '통과(팀장 최종 결정 필요)' if candidate_pass else '보류')
print('주의: 기존 전체 train 실험에 노출된 홀드아웃으로, 독립 외부 검증은 아닙니다.')

개발: 4960 홀드아웃: 1241 비율: 0.2001


,candidate,cv_mean,holdout_macro_f1
0,기존 XGB + LR,0.492355,0.509598
1,가중 XGB + LR,0.501080,0.499884


CV 변화 +0.0087 | 홀드아웃 변화 -0.0097
제출 후보 판단: 통과(팀장 최종 결정 필요)
주의: 기존 전체 train 실험에 노출된 홀드아웃으로, 독립 외부 검증은 아닙니다.


## 3. B 전체 Train OOF + Test 확률 생성 및 저장

**실행 전제:** 위 검증에서 `candidate_pass=True`여야 합니다. 전체 6,201행에 대해 팀 `get_folds` Group CV를 새로 계산해 각 행의 미학습 Fold 예측(OOF)을 만들고, 전체 Train으로 재학습한 모델로 Test 확률을 계산합니다. **이 OOF 점수는 홀드아웃 검증 점수와 별개**입니다.

`GENERATE_B_FILES=True`로 설정하면 파일을 생성합니다. 생성된 파일은 `experiments/preds/`에 저장하며 팀 공통 `save_predictions`도 호출합니다. 동일 파일명 덮어쓰기를 방지합니다.


In [7]:
GENERATE_B_FILES = True  # 검증 통과 후 전체 Train OOF / Test 예측 저장
B_NAME = 'ensemble_E_xgb_invsqrt_lr_5050_group_full_1110'

if not GENERATE_B_FILES:
    print('B 예측 파일 생성 비활성화')
else:
    if not candidate_pass:
        raise RuntimeError('B 후보가 CV/홀드아웃 통과 조건을 만족하지 않습니다. 파일 생성 중단')

    # 전체 Train 기준 팀 공통 Group CV. 개발셋 OOF와 혼동하지 않습니다.
    full_folds = list(get_folds(y, X=X_raw, classes=le.classes_))
    n_train, n_test = len(train), len(test)
    assert n_train == 6201 and n_test == 2546, (n_train, n_test)
    assert len(full_folds) == 5
    coverage = np.zeros(n_train, dtype=np.int32)
    for tr, va in full_folds:
        tr, va = np.asarray(tr), np.asarray(va)
        assert not np.intersect1d(tr, va).size
        coverage[va] += 1
    assert np.all(coverage == 1), '모든 Train 행은 정확히 한 번 검증 Fold에 포함돼야 합니다.'

    # 동일한 fold를 XGBoost와 LR에 적용; sample_weight는 각 학습 Fold만으로 계산
    print('전체 Train: weighted XGBoost E 5-Fold OOF')
    cv_xgb_full = run_cv(xgb, X_E, y, folds=full_folds, fit_params_fn=weighted_fit_params)
    print('전체 Train: LR E 5-Fold OOF')
    X_lr_full = pd.DataFrame.sparse.from_spmatrix(
        csr_matrix(X_E.to_numpy()), index=X_E.index, columns=X_E.columns
    )
    cv_lr_full = run_cv(lr, X_lr_full, y, folds=full_folds)
    p_xgb_oof = checked_proba(cv_xgb_full['oof_proba'], n_train)
    p_lr_oof = checked_proba(cv_lr_full['oof_proba'], n_train)
    b_oof = checked_proba((p_xgb_oof + p_lr_oof) / 2, n_train)

    # 전체 Train 재학습 → Test 예측 (OOF와 별도)
    print('전체 Train 재학습 → Test 확률')
    final_xgb = clone(xgb).fit(X_E, y, sample_weight=weight_array(y))
    final_lr = clone(lr).fit(csr_matrix(X_E.to_numpy()), y)
    assert np.array_equal(final_xgb.classes_, np.arange(K))
    assert np.array_equal(final_lr.classes_, np.arange(K))
    p_xgb_test = checked_proba(final_xgb.predict_proba(X_test_E), n_test)
    p_lr_test = checked_proba(final_lr.predict_proba(csr_matrix(X_test_E.to_numpy())), n_test)
    b_test = checked_proba((p_xgb_test + p_lr_test) / 2, n_test)

    # ID 및 클래스 순서를 명시한 공유용 CSV
    assert 'ID' in train.columns and 'ID' in test.columns
    assert train['ID'].notna().all() and test['ID'].notna().all()
    assert train['ID'].astype(str).is_unique and test['ID'].astype(str).is_unique
    cols = list(map(str, le.classes_))
    assert len(cols) == K and len(set(cols)) == K
    oof_df = pd.DataFrame(b_oof, columns=cols)
    test_df = pd.DataFrame(b_test, columns=cols)
    oof_df.insert(0, 'ID', train['ID'].astype(str).to_numpy())
    test_df.insert(0, 'ID', test['ID'].astype(str).to_numpy())
    assert oof_df.shape == (6201, K + 1) and test_df.shape == (2546, K + 1)

    out_dir = ROOT / 'experiments' / 'preds'
    out_dir.mkdir(parents=True, exist_ok=True)
    oof_path = out_dir / f'A_model__{B_NAME}__oof_with_ID.csv'
    test_path = out_dir / f'A_model__{B_NAME}__test_with_ID.csv'
    info_path = out_dir / f'A_model__{B_NAME}__info.json'
    # 이미 동일 실험 파일이 있으면 중단: 결과를 덮어쓰지 않음
    for path in (oof_path, test_path, info_path):
        if path.exists():
            raise FileExistsError(f'기존 결과 보호: {path}. 새 실험 이름 B_NAME을 사용하세요.')

    full_fold_scores = [macro_f1(y[va], np.argmax(b_oof[va], axis=1)) for _, va in full_folds]
    full_oof_score = macro_f1(y, np.argmax(b_oof, axis=1))
    cv_b_full = {
        'mean': float(np.mean(full_fold_scores)),
        'std': float(np.std(full_fold_scores)),
        'scores': list(map(float, full_fold_scores)),
        'oof': float(full_oof_score),
        'oof_proba': b_oof,
    }
    info = {
        'experiment': B_NAME,
        'description': 'weighted XGBoost E + balanced LR E 50:50',
        'train_rows': n_train, 'test_rows': n_test,
        'classes': cols, 'n_classes': K,
        'oof_method': '5-fold team get_folds Group CV, full train; out-of-fold only',
        'test_method': 'full-train refit of weighted XGB and LR; 50:50 mean',
        'full_train_cv_mean_macro_f1': cv_b_full['mean'],
        'full_train_oof_macro_f1': cv_b_full['oof'],
        'dev_cv_mean_macro_f1': float(cv_ens_weighted['mean']),
        'fixed_holdout_macro_f1': float(hold_weighted),
        'fixed_holdout_delta_vs_A': float(hold_delta),
        'note': 'Different CV populations; full-train OOF is not fixed-holdout evaluation.',
    }
    # 원자적 저장은 아니므로 팀 공유 전 파일 3개가 모두 생성되었는지 확인하세요.
    oof_df.to_csv(oof_path, index=False, encoding='utf-8-sig')
    test_df.to_csv(test_path, index=False, encoding='utf-8-sig')
    info_path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')

    # 팀의 표준 예측 저장 규약도 유지 (별도 파일 생성)
    save_predictions(
        owner='A_model', name=B_NAME,
        oof_proba=b_oof, test_proba=b_test,
        classes=le.classes_, cv=cv_b_full,
        notes='B weighted XGB E + LR E 50:50; full-train 5-fold Group CV OOF; full-train Test refit'
    )
    print('B 전체 Train Group CV 평균 Macro F1:', round(cv_b_full['mean'], 6))
    print('B 전체 Train OOF Macro F1:', round(full_oof_score, 6))
    print('저장 완료:', oof_path)
    print('저장 완료:', test_path)
    print('저장 완료:', info_path)
    display(oof_df.head(2))
    display(test_df.head(2))


전체 Train: weighted XGBoost E 5-Fold OOF
fold 1: Macro F1 = 0.4773
fold 2: Macro F1 = 0.5063
fold 3: Macro F1 = 0.4737
fold 4: Macro F1 = 0.5160
fold 5: Macro F1 = 0.4987
CV Macro F1 = 0.4944 (std 0.0164)
전체 Train: LR E 5-Fold OOF
fold 1: Macro F1 = 0.4934
fold 2: Macro F1 = 0.4819
fold 3: Macro F1 = 0.4738
fold 4: Macro F1 = 0.5161
fold 5: Macro F1 = 0.4949
CV Macro F1 = 0.4920 (std 0.0143)
전체 Train 재학습 → Test 확률
예측 저장: A_model__ensemble_E_xgb_invsqrt_lr_5050_group_full_1110 (CV 0.5127) → experiments/preds/
B 전체 Train Group CV 평균 Macro F1: 0.5127
B 전체 Train OOF Macro F1: 0.514265
저장 완료: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds\A_model__ensemble_E_xgb_invsqrt_lr_5050_group_full_1110__oof_with_ID.csv
저장 완료: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds\A_model__ensemble_E_xgb_invsqrt_lr_5050_group_full_1110__test_with_ID.csv
저장 완료: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\experiments\preds\A_model__ensemble_E_xgb_invsq

,ID,ACC,BLCA,BRCA,CESC,COAD,DLBC,GBMLGG,HNSC,KIPAN,...,PAAD,PCPG,PRAD,SARC,SKCM,STES,TGCT,THCA,THYM,UCEC
0,TRAIN_0000,0.036112,0.018914,0.051330,0.043299,0.026192,0.013578,0.055509,0.027581,0.098249,...,0.052993,0.004118,0.024046,0.020806,0.008890,0.091952,0.129223,0.002249,0.002384,0.020926
1,TRAIN_0001,0.006133,0.007427,0.031506,0.007617,0.036861,0.010918,0.235878,0.033860,0.023422,...,0.075589,0.006838,0.001905,0.265841,0.003394,0.059517,0.009573,0.001484,0.001011,0.015769


,ID,ACC,BLCA,BRCA,CESC,COAD,DLBC,GBMLGG,HNSC,KIPAN,...,PAAD,PCPG,PRAD,SARC,SKCM,STES,TGCT,THCA,THYM,UCEC
0,TEST_0000,0.025028,0.019877,0.040602,0.001778,0.080684,0.029519,0.024229,0.047219,0.058918,...,0.005808,0.000776,0.002442,0.005406,0.028809,0.453192,0.002216,0.000513,0.000869,0.016397
1,TEST_0001,0.002045,0.000522,0.002638,0.000805,0.011234,0.009089,0.001542,0.003753,0.000470,...,0.000225,0.000222,0.000372,0.001281,0.000294,0.861458,0.000384,0.000132,0.000221,0.096244


## 4. 실행 후 GitHub PR 체크

1. **홀드아웃 PR:** `scripts/fixed_holdout_manifest.py`, `data/splits/holdout_seed42.csv`, `data/splits/holdout_seed42_meta.json` (환자 ID 공유 권한 확인). 기존 CSV는 **재생성하지 않음**.
2. **B 실험 PR:** 이 노트북과 `experiments/preds/`의 B 산출물을 검토 후 추가. `git status`로 다른 팀원 파일이 포함되지 않았는지 확인.
3. B의 `oof_with_ID.csv`는 **Train 6201행**, `test_with_ID.csv`는 **Test 2546행**. 두 CSV 모두 첫 열 `ID`, 나머지 26개 열은 `le.classes_` 순서의 확률.
4. 제출용 라벨 CSV는 대회 `sample_submission.csv`의 정확한 형식을 확인한 뒤 별도로 작성해야 합니다. 확률 파일을 제출 형식이라고 가정하지 마세요.
5. 이 노트북의 전체 Train OOF는 **전체 6201행의 Group CV**이며, 앞서 평가한 개발셋 CV(4960행)와 직접 같은 수치가 아닙니다. 검증 홀드아웃은 과거 실험에 노출됐을 수 있습니다.


In [8]:
# B 최종 Test 확률 -> 예측 클래스 인덱스 -> 원본 SUBCLASS 라벨 -> 팀 표준 제출 CSV
from src.prediction import make_submission

if not candidate_pass:
    raise RuntimeError('B 모델의 개발 CV / 고정 홀드아웃 통과 여부를 먼저 확인하세요.')

# 동일 커널에서는 b_test 사용; 노트북을 다시 열었을 때는 저장된 팀 표준 확률 파일 사용
if 'b_test' in globals():
    submission_proba = np.asarray(b_test, dtype=np.float64)
else:
    saved_test = ROOT / 'experiments' / 'preds' / f'A_model__{B_NAME}__test.npy'
    if not saved_test.is_file():
        raise FileNotFoundError(f'B Test 확률이 없습니다: {saved_test}. 앞의 생성 셀을 실행하세요.')
    submission_proba = np.load(saved_test, allow_pickle=False)

assert submission_proba.shape == (len(test), len(le.classes_)), 'Test 행 수 또는 클래스 수가 일치하지 않습니다.'
assert np.isfinite(submission_proba).all(), '확률에 NaN 또는 무한대가 있습니다.'
assert np.allclose(submission_proba.sum(axis=1), 1, atol=1e-4), '확률 행 합계가 1이 아닙니다.'

submission_indices = np.argmax(submission_proba, axis=1)
submission_labels = le.inverse_transform(submission_indices)
submission_name = f'A_model__{B_NAME}__submission.csv'
submission_df = make_submission(test=test, pred_labels=submission_labels, file_name=submission_name)
assert len(submission_df) == len(test)
assert np.array_equal(submission_df['SUBCLASS'].astype(str).to_numpy(), submission_labels.astype(str))
print('제출 후보 파일 생성 완료:', ROOT / 'submissions' / submission_name)
print('실제 리더보드 제출은 팀장이 진행합니다.')
display(submission_df.head())


저장: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\submissions\A_model__ensemble_E_xgb_invsqrt_lr_5050_group_full_1110__submission.csv (2546행)
제출 후보 파일 생성 완료: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\submissions\A_model__ensemble_E_xgb_invsqrt_lr_5050_group_full_1110__submission.csv
실제 리더보드 제출은 팀장이 진행합니다.


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,LUSC
